# Insurance Claim Prediction
## 3. Data Preprocessing

### Objective
To clean, transform, and prepare the dataset for machine learning models.

## 3.1 Load Libraries and Data
* Load raw data
* import preprocessing libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [3]:
Train_data = pd.read_csv("Train_data.csv")

## 3.2 Feature Engineering- Building Age
* We want to extract useful information from Date_of_Occupancy

In [4]:
# Ensure date_of_occupancy is numeric(year)
Train_data['Date_of_Occupancy']=pd.to_numeric(Train_data['Date_of_Occupancy'], errors = 'coerce')

# create building_age
Train_data['Building_Age'] = Train_data['YearOfObservation'] - Train_data['Date_of_Occupancy']

## Reason:
* Age of the building can affect insurance claim probability
* Longer-lived buildings may have higher risk

## 3.3 Define Feature Groups
* We separate numerical and categorical features to process them differently

In [5]:
numerical_features = ['Insured_Period', 'Building Dimension', 'NumberOfWindows', 'Building_Age']

categorical_features = ['Residential', 'Building_Painted', 'Building_Fenced', 'Garden', 'Settlement',
                        'Building_Type', 'Geo_Code']

## 3.4 Create Pipelines for Preprocessing
### Numerical Pipeline
* Fill missing values with median
* Scale features with StandardScaler

In [6]:
num_pipeline = Pipeline([('imputer',
SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
                        ])

## categorical pipeline

* filling missing values with most frequent value

* one-hot encode categories

In [7]:
from sklearn.preprocessing import OneHotEncoder

cat_pipeline = Pipeline([('imputer',
SimpleImputer(strategy='most_frequent')),
    ('encoder',  
OneHotEncoder(handle_unknown='ignore'))
                        ])

## 3.5 combine pipelines using column transformer

In [8]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', num_pipeline,
numerical_features),
    ('cat', cat_pipeline,
categorical_features)
])

## Reason:
* Ensures preprocessing is applied automatically

* prevent leakage when using pipelines in modelling

## 3.6 Separate Features and Target

In [9]:
x = Train_data[numerical_features + categorical_features]
y = Train_data['Claim']

## 3.7: Apply Preprocessing

In [10]:
print("Numerical features:",
numerical_features)
print("Categorical features:",
categorical_features)

Numerical features: ['Insured_Period', 'Building Dimension', 'NumberOfWindows', 'Building_Age']
Categorical features: ['Residential', 'Building_Painted', 'Building_Fenced', 'Garden', 'Settlement', 'Building_Type', 'Geo_Code']


In [11]:
print(x.columns.tolist())

['Insured_Period', 'Building Dimension', 'NumberOfWindows', 'Building_Age', 'Residential', 'Building_Painted', 'Building_Fenced', 'Garden', 'Settlement', 'Building_Type', 'Geo_Code']


In [12]:
x['NumberOfWindows'].unique()

array(['   .', '4', '3', '2', '5', '>=10', '6', '7', '9', '8', '1'],
      dtype=object)

## 3.8 from my output
* NumberOfWindows is NOT numerical
* values are strings
* contains categories like '>=10'
* Not suitable for median/scaling

In [13]:
numerical_features.remove('NumberOfWindows')
categorical_features.append('NumberOfWindows')

In [14]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')

### Feature Engineering note:
During preprocessing, the NumberOfWindows feature was initially treated as numerical. However, exploratory analysis showed that it contained categorical string values(e.g '1', '2', '>=10'). To avoid incorrect numerical assumptions, this feature was reclassified as categorical and encoded accordingly. This improved preprocessing stability and model compatibility

In [15]:
X_processed = preprocessor.fit_transform(x)
print("Preprocessing complete. Processed feature shape:", X_processed.shape)
                        

Preprocessing complete. Processed feature shape: (7160, 1335)


## 3.9 result from preprocessing
* 7,160 rows
* 1,335 features after encoding
* OneHotEncoding expanded categorical features
* imputation + scaling worked

## Saved processed data

In [16]:
import pandas as pd

# Convert processed data to DataFrame
X_processed_df = pd.DataFrame(X_processed.toarray())

# Save features
X_processed_df.to_csv("X_processed.csv", index=False)

# Save target
y.to_csv("y.csv", index=False)

## Conclusion
The dataset is now clean, transformed, and ready for modeling

In [17]:
import joblib
joblib.dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']

In [18]:
print(x. columns)

Index(['Insured_Period', 'Building Dimension', 'NumberOfWindows',
       'Building_Age', 'Residential', 'Building_Painted', 'Building_Fenced',
       'Garden', 'Settlement', 'Building_Type', 'Geo_Code'],
      dtype='object')


In [19]:
Train_data['Geo_Code'].unique()

array(['1053', '1143', '1160', ..., '2B096', '2B353', nan], dtype=object)

In [20]:
Train_data.head()

,Customer Id,YearOfObservation,Insured_Period,Residential,Building_Painted,Building_Fenced,Garden,Settlement,Building Dimension,Building_Type,Date_of_Occupancy,NumberOfWindows,Geo_Code,Claim,Building_Age
0,H14663,2013,1.0,0,N,V,V,U,290.0,1,1960.0,.,1053,0,53.0
1,H2037,2015,1.0,0,V,N,O,R,490.0,1,1850.0,4,1053,0,165.0
2,H3802,2014,1.0,0,N,V,V,U,595.0,1,1960.0,.,1053,0,54.0
3,H3834,2013,1.0,0,V,V,V,U,2840.0,1,1960.0,.,1053,0,53.0
4,H5053,2014,1.0,0,V,N,O,R,680.0,1,1800.0,3,1053,0,214.0


In [21]:
import sklearn
print(sklearn.__version__)

1.4.2
